# ML-06 Feature Build

`00_ml_06_fb_author_contract.ipynb`에서 생성한 ML-06 input contract를 읽고 ML-06 feature artifacts를 생성한다.


## 00 Path Setup


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'Graph_AML' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_BUILD_DIR = PROJECT_ROOT / 'ml' / 'ml-06' / '00_feature_build'
if str(FEATURE_BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(FEATURE_BUILD_DIR))

PROJECT_ROOT, FEATURE_BUILD_DIR


(PosixPath('/mnt/d/AML_git/Graph_AML'),
 PosixPath('/mnt/d/AML_git/Graph_AML/ml/ml-06/00_feature_build'))

## 01 Contract Preflight and Build Configuration


In [2]:
from ml_06_fb_build import FeatureBuildConfig
from ml_06_fb_catalog import validate_input_contract
from ml_06_fb_io import DEFAULT_FB_OUTPUT_DIR, DEFAULT_INPUT_CONTRACT_PATH, DEFAULT_INPUT_PATH, DEFAULT_ML_INPUT_DIR

ARTIFACT_PREFIX = 'ml_06__r00'
SAMPLE_ROWS = None
OVERWRITE_OUTPUTS = False

source_columns = pq.read_schema(DEFAULT_INPUT_PATH).names
input_contract = pd.read_csv(DEFAULT_INPUT_CONTRACT_PATH, encoding='utf-8-sig', dtype={'used_in_ml': 'string'})
validation = validate_input_contract(input_contract, artifact_prefix=ARTIFACT_PREFIX, available_columns=list(source_columns))

config = FeatureBuildConfig(
    input_path=DEFAULT_INPUT_PATH,
    source_contract_path=DEFAULT_INPUT_CONTRACT_PATH,
    fb_output_dir=DEFAULT_FB_OUTPUT_DIR,
    ml_input_dir=DEFAULT_ML_INPUT_DIR,
    artifact_prefix=ARTIFACT_PREFIX,
    sample_rows=SAMPLE_ROWS,
    overwrite=OVERWRITE_OUTPUTS,
)

print('input_contract_path:', DEFAULT_INPUT_CONTRACT_PATH)
print('contract_rows      :', validation.total_rows)
print('selected_count     :', validation.selected_count)
print('ratio_base_count   :', len(validation.ratio_base_columns))
print('sample_rows        :', SAMPLE_ROWS)
print('overwrite          :', OVERWRITE_OUTPUTS)
display(input_contract.loc[input_contract['used_in_ml'].astype(str).str.strip() == 'TRUE'].head(30))


input_contract_path: /mnt/d/AML_git/Graph_AML/ml/ml-06/fb_inputs/r00/ml_06__r00_fb_input_feature_contract.csv
contract_rows      : 317
selected_count     : 236
ratio_base_count   : 21
sample_rows        : None
overwrite          : False


,contract_version,artifact_prefix,column_name,used_in_ml,source_column,column_origin,encoding,build_action,build_in_fb,xgb_feature_type,...,observed_dtype,missing_count,missing_rate,unknown_category_count,fit_split,note,ml_06_policy,ml_06_ratio_base_column,ml_06_ratio_variant,recency_sentinel_policy
17,1,ml_06__r00,amount__current__value,TRUE,amount__current__value,preprocessing,passthrough,carry_forward,False,q,...,float64,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
18,1,ml_06__r00,amount__current__log1p,TRUE,amount__current__log1p,preprocessing,passthrough,carry_forward,False,q,...,float64,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
27,1,ml_06__r00,amount_received__current__value,TRUE,amount_received__current__value,preprocessing,passthrough,carry_forward,False,q,...,float64,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
28,1,ml_06__r00,amount_received__current__log1p,TRUE,amount_received__current__log1p,preprocessing,passthrough,carry_forward,False,q,...,float64,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
36,1,ml_06__r00,timehist__sender__out__tx_count__count__w1h,TRUE,timehist__sender__out__tx_count__count__w1h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
37,1,ml_06__r00,timehist__sender__out__amount__sum__w1h,TRUE,timehist__sender__out__amount__sum__w1h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
38,1,ml_06__r00,timehist__sender__out__amount__mean__w1h,TRUE,timehist__sender__out__amount__mean__w1h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
39,1,ml_06__r00,timehist__sender__out__amount__std__w1h,TRUE,timehist__sender__out__amount__std__w1h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
40,1,ml_06__r00,timehist__sender__out__amount__max__w1h,TRUE,timehist__sender__out__amount__max__w1h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN
41,1,ml_06__r00,timehist__sender__out__tx_count__count__w6h,TRUE,timehist__sender__out__tx_count__count__w6h,preprocessing,passthrough,carry_forward,False,q,...,float32,NaN,NaN,NaN,NaN,source parquet column,NaN,NaN,NaN,NaN


## 02 Execute Build


In [3]:
from ml_06_fb_build import build_features

result = build_features(config)
result.row_counts, len(result.generated_columns), len(result.base_ratio_columns)


({'all': 5077237, 'train': 3046186, 'val': 1015633, 'test': 1015418}, 42, 21)

## 03 Review Reports


In [4]:
result.recency_report


,column_name,policy,changed,row_count,changed_count,before_min,before_max,before_negative_count,before_minus1_count,before_zero_count,after_min,after_max,after_negative_count,after_minus1_count,after_zero_count,note
0,recency__sender__out__seconds_since_last,minus1_to_zero,TRUE,5077237,502577,-1.0,863700.0,502577,502577,0,0.0,863700.0,0,0,502577,first transaction meaning preserved by existin...
1,recency__receiver__in__seconds_since_last,minus1_to_zero,TRUE,5077237,424939,-1.0,862140.0,424939,424939,0,0.0,862140.0,0,0,424939,first transaction meaning preserved by existin...
2,pair__sender_receiver__forward__seconds_since_...,audit_only_no_value_change,FALSE,5077237,0,0.0,861120.0,0,0,1027487,0.0,861120.0,0,0,1027487,
3,passflow__sender__in_then_out__seconds_since_l...,audit_only_no_value_change,FALSE,5077237,0,0.0,3600.0,0,0,4585253,0.0,3600.0,0,0,4585253,
4,passflow__sender__in_then_out__seconds_since_l...,audit_only_no_value_change,FALSE,5077237,0,0.0,21600.0,0,0,3700206,0.0,21600.0,0,0,3700206,
5,passflow__sender__in_then_out__seconds_since_l...,audit_only_no_value_change,FALSE,5077237,0,0.0,86400.0,0,0,2016381,0.0,86400.0,0,0,2016381,


In [5]:
result.ratio_reuse_report.head(20)


,base_column,transform,output_column,action,reason
0,amount__paid_recv_ratio,log1p,amount__paid_recv_ratio__log1p,created,created pure log1p output
1,amount__paid_recv_ratio,clip_train_p9999,amount__paid_recv_ratio__clip_train_p9999,created,created train p99.99 clipped output
2,timehist__sender__out__amount__cur_vs_mean_rat...,log1p,timehist__sender__out__amount__cur_vs_mean_rat...,created,created pure log1p output
3,timehist__sender__out__amount__cur_vs_mean_rat...,clip_train_p9999,timehist__sender__out__amount__cur_vs_mean_rat...,created,created train p99.99 clipped output
4,timehist__sender__out__amount__cur_vs_mean_rat...,log1p,timehist__sender__out__amount__cur_vs_mean_rat...,created,created pure log1p output
5,timehist__sender__out__amount__cur_vs_mean_rat...,clip_train_p9999,timehist__sender__out__amount__cur_vs_mean_rat...,created,created train p99.99 clipped output
6,timehist__sender__out__amount__cur_vs_mean_rat...,log1p,timehist__sender__out__amount__cur_vs_mean_rat...,created,created pure log1p output
7,timehist__sender__out__amount__cur_vs_mean_rat...,clip_train_p9999,timehist__sender__out__amount__cur_vs_mean_rat...,created,created train p99.99 clipped output
8,timehist__receiver__in__amount__cur_vs_mean_ra...,log1p,timehist__receiver__in__amount__cur_vs_mean_ra...,created,created pure log1p output
9,timehist__receiver__in__amount__cur_vs_mean_ra...,clip_train_p9999,timehist__receiver__in__amount__cur_vs_mean_ra...,created,created train p99.99 clipped output
